### Robot Tracking with ID Detection

This notebook is very similar to botsort.ipynb however, I am also using frame detection to find the IDs of the robots involved in the match and align them with the IDs assigned by botsort. 

In [1]:
!pip install ultralytics opencv-python easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 30.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 131.5 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 218.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 143.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 403.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [easyocr]9/10 [easyocr]mage]-headless]


In [2]:
import os
from pathlib import Path
from scipy.optimize import linear_sum_assignment

REPO_ROOT = Path(os.getcwd()).parent  

In [3]:
# Finding path to best model
for p in REPO_ROOT.rglob("best_tuned_yolov8.pt"):
    print(p)

/work/classtmp/kmwendl/FIRST-Robotics-Competition-Data-Challenge/yolov8_model/best_tuned_yolov8.pt


In [4]:
VIDEO_NAME = 'cropped_Qualification 45 - 2025 Central Missouri Regional.mp4'
VIDEO_PATH = "/work/classtmp/FIRST-Robotics-Competition-Data-Challenge-Videos/cropped_videos/cropped_Qualification 45 - 2025 Central Missouri Regional.mp4"

# Change Model Path to be best model from above in line 2! 
MODEL_PATH = REPO_ROOT / "yolov8_model" / "best_tuned_yolov8.pt"

# Custom tracker 
CUSTOM_TRACKER_PATH = REPO_ROOT / "trackers" / "botsort_custom.yaml"

# Checking to see if file exist! 
print("Model:", MODEL_PATH)
print("Video:", VIDEO_PATH)
print("Tracker:", CUSTOM_TRACKER_PATH)

print("Model exists:", MODEL_PATH.exists())

Model: /work/classtmp/kmwendl/FIRST-Robotics-Competition-Data-Challenge/yolov8_model/best_tuned_yolov8.pt
Video: /work/classtmp/FIRST-Robotics-Competition-Data-Challenge-Videos/cropped_videos/cropped_Qualification 45 - 2025 Central Missouri Regional.mp4
Tracker: /work/classtmp/kmwendl/FIRST-Robotics-Competition-Data-Challenge/trackers/botsort_custom.yaml
Model exists: True


In [5]:
from ultralytics import YOLO
import cv2
import numpy as np
import easyocr
from collections import defaultdict

# Load models
robot_model = YOLO(MODEL_PATH)
reader = easyocr.Reader(['en'], gpu=True)

# Mapping: botsort_id -> detected robot number
track_id_to_robot_id = {}

# Confidence tracking for stability
id_votes = defaultdict(lambda: defaultdict(int))

def extract_robot_id(crop):
    """
    Use OCR to extract number from robot crop
    """
    results = reader.readtext(crop)

    for (bbox, text, conf) in results:
        # Keep only numeric detections
        if text.isdigit() and conf > 0.4:
            return text

    return None

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [10]:
def run_botsort_with_id(video_path, output_path=None):
    cap = cv2.VideoCapture(video_path)

    width = int(cap.get(3))
    height = int(cap.get(4))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    if output_path:
        out = cv2.VideoWriter(
            output_path,
            cv2.VideoWriter_fourcc(*'mp4v'),
            fps,
            (width, height)
        )

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Run tracking
        results = robot_model.track(
            frame,
            persist=True,
            tracker="botsort.yaml"
        )

        if results[0].boxes.id is None:
            continue

        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)

        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = map(int, box)

            crop = frame[y1:y2, x1:x2]

            detected_id = extract_robot_id(crop)

            # Voting system for stability
            if detected_id:
                id_votes[track_id][detected_id] += 1

                # Lock ID once confident
                best_id = max(id_votes[track_id], key=id_votes[track_id].get)

                if id_votes[track_id][best_id] > 5:
                    track_id_to_robot_id[track_id] = best_id

            # Use mapped ID if exists
            label = track_id_to_robot_id.get(track_id, f"Tracking {track_id}")

            # Draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(frame, f"Robot {label}", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

        if output_path:
            out.write(frame)

        pass

    cap.release()
    if output_path:
        out.release()
    

In [11]:
run_botsort_with_id(
    video_path = VIDEO_PATH,
    output_path = REPO_ROOT / "tracking_output" / "output_with_ids.mp4"
)


0: 160x640 (no detections), 5.5ms
Speed: 0.8ms preprocess, 5.5ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.9ms
Speed: 0.6ms preprocess, 4.9ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.8ms
Speed: 0.6ms preprocess, 4.8ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.8ms
Speed: 0.6ms preprocess, 4.8ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.8ms
Speed: 0.6ms preprocess, 4.8ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.7ms
Speed: 0.6ms preprocess, 4.7ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.8ms
Speed: 0.6ms preprocess, 4.8ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.8ms
Speed: 0.6ms preprocess, 4.8ms inference, 0.4ms 

0: 160x640 (no detections), 4.7ms
Speed: 0.6ms preprocess, 4.7ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.7ms
Speed: 0.6ms preprocess, 4.7ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.7ms
Speed: 0.6ms preprocess, 4.7ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.7ms
Speed: 0.6ms preprocess, 4.7ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.7ms
Speed: 0.6ms preprocess, 4.7ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.7ms
Speed: 0.6ms preprocess, 4.7ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.7ms
Speed: 0.6ms preprocess, 4.7ms inference, 0.4ms postprocess per image at shape (1, 3, 160, 640)

0: 160x640 (no detections), 4.6ms
Speed: 0.6ms preprocess, 4.6ms inference, 0.4ms p